# Earnings Correlation Trading Strategy

This notebook demonstrates how to use the earnings correlation analyzer to generate trading signals based on correlated company earnings calls.

## Strategy Overview

The strategy is based on the following hypothesis:
- Companies in the same sector often show similar price movements after earnings calls
- By analyzing historical price movements of correlated companies after their earnings, we can predict how a target stock will move
- This creates a leading indicator: when correlated companies report earnings, we can position ourselves before the target company's earnings

In [ ]:
import os
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

# Add src directory to path
sys.path.insert(0, os.path.abspath('../src'))
from earnings_correlation_analyzer import (
    analyze_earnings_correlation,
    load_summary_data,
    load_chart_data,
    get_sector
)

# Set up data directory
data_dir = os.path.abspath('../data')

## Example 1: Analyze Apple (AAPL)

In [ ]:
# Analyze AAPL with 5-day horizon
results_aapl = analyze_earnings_correlation(data_dir, 'AAPL', days_after=5)

# Display results
print(f"\nSignal: {results_aapl.get('signal')}")
print(f"Confidence: {results_aapl.get('confidence', 0):.1f}%")
print(f"Average historical change: {results_aapl.get('avg_price_change', 0):.2f}%")

## Example 2: Compare Multiple Tech Stocks

In [ ]:
# Analyze multiple tech stocks
tech_stocks = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'META']
tech_results = []

for symbol in tech_stocks:
    print(f"\nAnalyzing {symbol}...")
    result = analyze_earnings_correlation(data_dir, symbol, days_after=5)
    if 'error' not in result:
        tech_results.append({
            'Symbol': symbol,
            'Signal': result['signal'],
            'Confidence': result['confidence'],
            'Avg Change': result['avg_price_change'],
            'Positive Ratio': result['positive_ratio'] * 100
        })

# Create comparison DataFrame
df_tech = pd.DataFrame(tech_results)
df_tech

## Example 3: Analyze Financial Sector

In [ ]:
# Analyze financial stocks
financial_stocks = ['JPM', 'BAC', 'WFC', 'GS', 'MS']
financial_results = []

for symbol in financial_stocks:
    print(f"\nAnalyzing {symbol}...")
    result = analyze_earnings_correlation(data_dir, symbol, days_after=5)
    if 'error' not in result:
        financial_results.append({
            'Symbol': symbol,
            'Signal': result['signal'],
            'Confidence': result['confidence'],
            'Avg Change': result['avg_price_change'],
            'Positive Ratio': result['positive_ratio'] * 100
        })

# Create comparison DataFrame
df_financial = pd.DataFrame(financial_results)
df_financial

## Example 4: Visualize Results

In [ ]:
# Combine all results
all_results = tech_results + financial_results
df_all = pd.DataFrame(all_results)

# Plot average price changes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Average price change by stock
colors = ['green' if x > 0 else 'red' for x in df_all['Avg Change']]
ax1.barh(df_all['Symbol'], df_all['Avg Change'], color=colors, alpha=0.7)
ax1.set_xlabel('Average Price Change (%)')
ax1.set_title('Average Price Change After Correlated Earnings (5 days)')
ax1.axvline(x=0, color='black', linestyle='--', linewidth=0.5)
ax1.grid(axis='x', alpha=0.3)

# Plot 2: Confidence by stock
signal_colors = {'BUY': 'green', 'SELL': 'red', 'NEUTRAL': 'gray'}
colors2 = [signal_colors.get(s, 'gray') for s in df_all['Signal']]
ax2.barh(df_all['Symbol'], df_all['Confidence'], color=colors2, alpha=0.7)
ax2.set_xlabel('Confidence Score')
ax2.set_title('Trading Signal Confidence')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## Example 5: Test Different Time Horizons

In [ ]:
# Test different time horizons for a single stock
symbol = 'AAPL'
horizons = [1, 3, 5, 10, 20]
horizon_results = []

for days in horizons:
    print(f"\nAnalyzing {symbol} with {days}-day horizon...")
    result = analyze_earnings_correlation(data_dir, symbol, days_after=days)
    if 'error' not in result:
        horizon_results.append({
            'Days': days,
            'Avg Change': result['avg_price_change'],
            'Std Dev': result['std_price_change'],
            'Positive Ratio': result['positive_ratio'] * 100
        })

df_horizons = pd.DataFrame(horizon_results)
df_horizons

## Example 6: Export Results for Trading

In [ ]:
# Save results to JSON for use in trading system
output_file = '../data/earnings_signals.json'

trading_signals = {}
for symbol in tech_stocks + financial_stocks:
    result = analyze_earnings_correlation(data_dir, symbol, days_after=5)
    if 'error' not in result:
        trading_signals[symbol] = {
            'signal': result['signal'],
            'confidence': result['confidence'],
            'avg_change': result['avg_price_change'],
            'upcoming_earnings': result.get('upcoming_earnings', [])[:3]
        }

with open(output_file, 'w') as f:
    json.dump(trading_signals, f, indent=4)

print(f"\nTrading signals saved to: {output_file}")

## Conclusion

The earnings correlation strategy provides a systematic way to:
1. Identify sector-wide trends based on earnings calls
2. Generate leading indicators by tracking correlated companies
3. Quantify confidence based on historical performance
4. Time trades around earnings seasons

### Key Insights:
- Technology stocks often show neutral patterns with high variability
- Financial stocks can show more pronounced sector-wide movements
- Shorter time horizons (1-3 days) reduce noise but may miss larger moves
- Longer horizons (10-20 days) capture more of the trend but add market noise

### Next Steps:
1. Backtest the strategy with historical data
2. Implement position sizing based on confidence scores
3. Add risk management (stop-loss, take-profit)
4. Monitor upcoming earnings calendars for correlated companies